In [1]:
import os
import time
import requests
import json
from pathlib import Path
from dotenv import load_dotenv

In [2]:
repo_root = Path.cwd().parent
load_dotenv(repo_root / ".env")

BASE_URL = "https://api.jolpi.ca/ergast/f1"

In [3]:
url = f"{BASE_URL}/2026/results.json"
response = requests.get(url)
print(response.status_code)
print(response.url)
data = response.json()
print(json.dumps(data, indent=2))


200
https://api.jolpi.ca/ergast/f1/2026/results.json
{
  "MRData": {
    "xmlns": "",
    "series": "f1",
    "url": "https://api.jolpi.ca/ergast/f1/2026/results.json",
    "limit": "30",
    "offset": "0",
    "total": "44",
    "RaceTable": {
      "season": "2026",
      "Races": [
        {
          "season": "2026",
          "round": "1",
          "url": "https://en.wikipedia.org/wiki/2026_Australian_Grand_Prix",
          "raceName": "Australian Grand Prix",
          "Circuit": {
            "circuitId": "albert_park",
            "url": "https://en.wikipedia.org/wiki/Albert_Park_Circuit",
            "circuitName": "Albert Park Grand Prix Circuit",
            "Location": {
              "lat": "-37.8497",
              "long": "144.968",
              "locality": "Melbourne",
              "country": "Australia"
            }
          },
          "date": "2026-03-08",
          "time": "04:00:00Z",
          "Results": [
            {
              "number": "63",
       

In [4]:
data = response.json()["MRData"]
total  = int(data["total"])
limit  = int(data["limit"])
offset = int(data["offset"])

print(f"Total records : {total}")
print(f"Limit         : {limit}")
print(f"Offset        : {offset}")
print(f"Pages needed  : {-(-total // limit)}")
print(f"More pages?   : {offset + limit < total}")

Total records : 44
Limit         : 30
Offset        : 0
Pages needed  : 2
More pages?   : True


In [5]:
races = data["RaceTable"]["Races"]
first_race = races[0]
for result in first_race["Results"][:3]:
    driver = result["Driver"]
    print(f"P{result['position']}: {driver['givenName']} {driver['familyName']} ({result['Constructor']['name']})")


P1: George Russell (Mercedes)
P2: Andrea Kimi Antonelli (Mercedes)
P3: Charles Leclerc (Ferrari)


In [6]:
url = f"{BASE_URL}/2026/driverStandings.json"
response = requests.get(url)
data = response.json()["MRData"]
print(json.dumps(data, indent=2))

{
  "xmlns": "",
  "series": "f1",
  "url": "https://api.jolpi.ca/ergast/f1/2026/driverstandings.json",
  "limit": "30",
  "offset": "0",
  "total": "22",
  "StandingsTable": {
    "season": "2026",
    "round": "2",
    "StandingsLists": [
      {
        "season": "2026",
        "round": "2",
        "DriverStandings": [
          {
            "position": "1",
            "positionText": "1",
            "points": "51",
            "wins": "1",
            "Driver": {
              "driverId": "russell",
              "permanentNumber": "63",
              "code": "RUS",
              "url": "http://en.wikipedia.org/wiki/George_Russell_(racing_driver)",
              "givenName": "George",
              "familyName": "Russell",
              "dateOfBirth": "1998-02-15",
              "nationality": "British"
            },
            "Constructors": [
              {
                "constructorId": "mercedes",
                "url": "https://en.wikipedia.org/wiki/Mercedes-Benz_in

In [7]:
standings = data["StandingsTable"]["StandingsLists"][0]["DriverStandings"]
for s in standings[:5]:
    driver = s["Driver"]
    print(f"P{s['position']}: {driver['givenName']} {driver['familyName']} ({s['points']} pts)")

P1: George Russell (51 pts)
P2: Andrea Kimi Antonelli (47 pts)
P3: Charles Leclerc (34 pts)
P4: Lewis Hamilton (33 pts)
P5: Oliver Bearman (17 pts)


In [8]:
url = f"{BASE_URL}/2026/1/pitstops.json"
response = requests.get(url)
data = response.json()["MRData"]
print(json.dumps(data, indent=2))

{
  "xmlns": "",
  "series": "f1",
  "url": "https://api.jolpi.ca/ergast/f1/2026/1/pitstops.json",
  "limit": "30",
  "offset": "0",
  "total": "32",
  "RaceTable": {
    "season": "2026",
    "round": "1",
    "Races": [
      {
        "season": "2026",
        "round": "1",
        "url": "https://en.wikipedia.org/wiki/2026_Australian_Grand_Prix",
        "raceName": "Australian Grand Prix",
        "Circuit": {
          "circuitId": "albert_park",
          "url": "https://en.wikipedia.org/wiki/Albert_Park_Circuit",
          "circuitName": "Albert Park Grand Prix Circuit",
          "Location": {
            "lat": "-37.8497",
            "long": "144.968",
            "locality": "Melbourne",
            "country": "Australia"
          }
        },
        "date": "2026-03-08",
        "time": "04:00:00Z",
        "PitStops": [
          {
            "driverId": "colapinto",
            "lap": "9",
            "stop": "1",
            "time": "15:16:40",
            "duration"

In [19]:
import sys
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))
from ingestion.jolpica.helper import pagination_helper, logging_setup

logging_setup()
pages = pagination_helper("2025/results")
print(f"Pages returned: {len(pages)}")
print(f"Total field from first page: {pages[0]['total']}")

[2026-03-17 13:10:17] INFO [ingestion.jolpica.helper] 2025/results: offset 0/479
[2026-03-17 13:10:17] INFO [ingestion.jolpica.helper] 2025/results: complete (1 page(s), 479 total records)


offset=0, limit=100, total=479
records in this page: 5


[2026-03-17 13:10:18] INFO [ingestion.jolpica.helper] 2025/results: offset 100/479
[2026-03-17 13:10:18] INFO [ingestion.jolpica.helper] 2025/results: complete (2 page(s), 479 total records)


offset=100, limit=100, total=479
records in this page: 6


[2026-03-17 13:10:20] INFO [ingestion.jolpica.helper] 2025/results: offset 200/479
[2026-03-17 13:10:20] INFO [ingestion.jolpica.helper] 2025/results: complete (3 page(s), 479 total records)


offset=200, limit=100, total=479
records in this page: 6


[2026-03-17 13:10:21] INFO [ingestion.jolpica.helper] 2025/results: offset 300/479
[2026-03-17 13:10:21] INFO [ingestion.jolpica.helper] 2025/results: complete (4 page(s), 479 total records)


offset=300, limit=100, total=479
records in this page: 6


[2026-03-17 13:10:22] INFO [ingestion.jolpica.helper] 2025/results: offset 400/479


offset=400, limit=100, total=479
records in this page: 4
break check: 400 + 100 >= 479 → True
Pages returned: 5
Total field from first page: 479


In [20]:
url = f"{BASE_URL}/2026/constructorStandings.json"
response = requests.get(url)
data = response.json()["MRData"]
print(json.dumps(data, indent=2))

{
  "xmlns": "",
  "series": "f1",
  "url": "https://api.jolpi.ca/ergast/f1/2026/constructorstandings.json",
  "limit": "30",
  "offset": "0",
  "total": "11",
  "StandingsTable": {
    "season": "2026",
    "round": "2",
    "StandingsLists": [
      {
        "season": "2026",
        "round": "2",
        "ConstructorStandings": [
          {
            "position": "1",
            "positionText": "1",
            "points": "98",
            "wins": "2",
            "Constructor": {
              "constructorId": "mercedes",
              "url": "https://en.wikipedia.org/wiki/Mercedes-Benz_in_Formula_One",
              "name": "Mercedes",
              "nationality": "German"
            }
          },
          {
            "position": "2",
            "positionText": "2",
            "points": "67",
            "wins": "0",
            "Constructor": {
              "constructorId": "ferrari",
              "url": "https://en.wikipedia.org/wiki/Scuderia_Ferrari",
              